In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/creditcard.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFraud cases: {df['Class'].sum()} ({df['Class'].mean()*100:.3f}%)")
print(f"Legitimate cases: {(df['Class']==0).sum()}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nFirst few rows:")
df.head()

Shape: (284807, 31)

Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Fraud cases: 492 (0.173%)
Legitimate cases: 284315

Missing values: 0

First few rows:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
import numpy as np

# Split features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Train/test split — stratified to preserve fraud ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} transactions")
print(f"Test set: {X_test.shape[0]} transactions")
print(f"Fraud in training: {y_train.sum()} cases")
print(f"Fraud in test: {y_test.sum()} cases")

# scale_pos_weight handles class imbalance
# = number of legitimate / number of fraud cases
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight: {scale_pos_weight:.1f}")

# Start MLflow run
mlflow.set_experiment("FinStreamAI-Fraud-Detection")

with mlflow.start_run(run_name="XGBoost-v1"):
    params = {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    # Predictions
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    # Metrics
    auc = roc_auc_score(y_test, probs)

    # Log to MLflow
    mlflow.log_params(params)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(model, "fraud_model")

    print(f"\nROC-AUC Score: {auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Legitimate', 'Fraud']))

Training set: 227845 transactions
Test set: 56962 transactions
Fraud in training: 394 cases
Fraud in test: 98 cases

scale_pos_weight: 577.3


2026/03/10 14:50:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/10 14:50:37 INFO mlflow.store.db.utils: Updating database tables
2026/03/10 14:50:41 INFO mlflow.tracking.fluent: Experiment with name 'FinStreamAI-Fraud-Detection' does not exist. Creating a new experiment.
2026/03/10 14:51:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 14:51:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



ROC-AUC Score: 0.9747

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.78      0.84      0.81        98

    accuracy                           1.00     56962
   macro avg       0.89      0.92      0.90     56962
weighted avg       1.00      1.00      1.00     56962



In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score

# Train Isolation Forest on legitimate transactions only
# It learns what "normal" looks like, then flags anomalies
X_train_legit = X_train[y_train == 0]

with mlflow.start_run(run_name="IsolationForest-v1"):
    iso_params = {
        'contamination': 0.002,  # expected fraud rate ~0.2%
        'random_state': 42,
        'n_estimators': 100
    }

    iso_model = IsolationForest(**iso_params)
    iso_model.fit(X_train_legit)

    # Isolation Forest returns -1 for anomaly, 1 for normal
    # Convert to 0/1 to match our Class labels
    iso_preds = iso_model.predict(X_test)
    iso_preds = (iso_preds == -1).astype(int)

    iso_scores = iso_model.score_samples(X_test)
    iso_auc = roc_auc_score(y_test, -iso_scores)

    mlflow.log_params(iso_params)
    mlflow.log_metric("roc_auc", iso_auc)

    print(f"Isolation Forest ROC-AUC: {iso_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, iso_preds, target_names=['Legitimate', 'Fraud']))
    print("\n--- Model Comparison ---")
    print(f"XGBoost ROC-AUC:          0.9747")
    print(f"Isolation Forest ROC-AUC: {iso_auc:.4f}")
    print(f"\nWinner: {'XGBoost' if 0.9747 > iso_auc else 'Isolation Forest'}")

Isolation Forest ROC-AUC: 0.9534

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.22      0.35      0.27        98

    accuracy                           1.00     56962
   macro avg       0.61      0.67      0.63     56962
weighted avg       1.00      1.00      1.00     56962


--- Model Comparison ---
XGBoost ROC-AUC:          0.9747
Isolation Forest ROC-AUC: 0.9534

Winner: XGBoost


In [4]:
import pickle
import os

# Save model to models/ folder
model_path = '../models/fraud_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# Save feature names so API knows expected input order
feature_names = list(X.columns)
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print(f"Model saved to {model_path}")
print(f"Features: {len(feature_names)} columns")
print(f"Feature names: {feature_names}")

Model saved to ../models/fraud_model.pkl
Features: 30 columns
Feature names: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
